# Public data experiments
This is the notebook to run the public data experiment

In [1]:
import sys
from pathlib import Path

# Add the workspace root to sys.path
workspace_root = Path("/Users/eduardohome/workspace/benchmark_credit_scoring")
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

# Now you can import from src
from src.utils.csv_config import read_csv_by_name, create_imbalanced_datasets, merge_xai_results
from src.model.functions import save_model, load_model, get_split_data
from src.xai.functions import get_train_iterations_shap, compute_xai_metrics_from_shap
from src.graph.functions import plot_model_boxplot, plot_imbalance_boxplot_grid

try:
    from src.model.functions import get_train_model
except Exception as exc:
    # provide helpful message if import fails
    print("Failed to import model utilities:", exc)
    print("This often means a missing dependency such as libomp for LightGBM.")
    print("On macOS you can install it with `brew install libomp` and then restart the kernel.")
    raise

# General imports
import pandas as pd
from pathlib import Path

In [2]:
# -------------------------
# Settings
# -------------------------
data_dir = Path("/Users/eduardohome/workspace/benchmark_credit_scoring/data")
#datasets_to_run = ["australian", "german", "japan", "kgccdea", "taiwan", "th02", "kggmsc"]
datasets_to_run = ["australian", "german", "japan", "kgccdea", "th02"]
target_col = "target"

# Edit this list whenever you want more/less imbalance datasets
imbalance_levels = [10, 15, 20, 25, 30, 35, 40, 45, 50]
imbalance_levels = [10, 15, 20, 25]
n_iterations = 100
percentage = 90

# list of estimators understood by functions.get_model
model_choices = ["XGBoost", "LightGBM", "RandomForest", "BoostDT", "BoostSGB", "BagNN", "ANN"] #, "AdaBoost-DT"
product = "default"

## Interpretability metrics

In [3]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable

import pandas as pd


# =============================================================================
# Settings
# =============================================================================

@dataclass(frozen=True)
class Settings:
    data_dir: Path = Path("/Users/eduardohome/workspace/benchmark_credit_scoring/data")
    datasets_to_run: list[str] = field(
        default_factory=lambda: ["australian", "german", "japan", "kgccdea", "th02"]
    )
    target_col: str = "target"
    imbalance_levels: list[int] = field(default_factory=lambda: [10, 15, 20, 25])
    n_iterations: int = 100
    percentage: int = 90
    model_choices: list[str] = field(
        default_factory=lambda: [
            "XGBoost",
            "LightGBM",
            "RandomForest",
            "BoostDT",
            "BoostSGB",
            "BagNN",
            "ANN",
        ]
    )
    product: str = "default"
    xai_file_types: list[str] = field(
        default_factory=lambda: ["resume_local", "resume_global"]
    )
    normalize_xai_outputs: bool = True


# =============================================================================
# Generic helpers
# =============================================================================

def dataset_variants(dataset_name: str, imbalance_levels: Iterable[int]) -> list[str]:
    return [dataset_name] + [f"{dataset_name}_{level}" for level in imbalance_levels]


def get_imbalance_label(dataset_name: str, dataset_variant: str) -> str | int:
    if dataset_variant == dataset_name:
        return "original"
    return dataset_variant.replace(f"{dataset_name}_", "")


def output_dir(settings: Settings) -> Path:
    path = settings.data_dir / "output"
    path.mkdir(parents=True, exist_ok=True)
    return path


def dataset_output_dir(settings: Settings, dataset_name: str) -> Path:
    path = output_dir(settings) / dataset_name
    path.mkdir(parents=True, exist_ok=True)
    return path


def xai_dir(settings: Settings, dataset_name: str, model: str, dataset_variant: str) -> Path:
    return dataset_output_dir(settings, dataset_name) / "xai" / model / dataset_variant


def metrics_dir(settings: Settings, dataset_name: str, model: str, dataset_variant: str) -> Path:
    path = xai_dir(settings, dataset_name, model, dataset_variant) / "metrics"
    path.mkdir(parents=True, exist_ok=True)
    return path


def shap_values_dir(settings: Settings, dataset_name: str, model: str, dataset_variant: str) -> Path:
    return xai_dir(settings, dataset_name, model, dataset_variant) / "shap_values"


def has_required_columns(df: pd.DataFrame, required_cols: set[str], context: str) -> bool:
    missing = required_cols - set(df.columns)
    if missing:
        print(f"[SKIP] {context}: missing columns {sorted(missing)}")
        return False
    return True


def rank_within_groups(
    df: pd.DataFrame,
    group_cols: list[str],
    rank_config: dict[str, bool],
) -> pd.DataFrame:
    """
    Add dense ranks for several metrics.

    rank_config maps metric name to ascending:
    - ascending=True means lower is better.
    - ascending=False means higher is better.
    """
    df = df.copy()

    for metric, ascending in rank_config.items():
        df[f"rank_{metric}"] = (
            df.groupby(group_cols)[metric]
            .rank(method="dense", ascending=ascending)
        )

    return df


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"[OK] Saved: {path}")


# =============================================================================
# 1. Compute XAI metrics from saved SHAP files
# =============================================================================

def compute_and_save_xai_metrics(settings: Settings) -> None:
    """
    Loads saved SHAP values, computes XAI metrics, and saves:
    - detail_local
    - resume_local
    - resume_global

    Requires compute_xai_metrics_from_shap(df_shap) to be available in scope.
    """
    for dataset_name in settings.datasets_to_run:
        print_dataset_header("XAI metrics", dataset_name)

        for dataset_variant in dataset_variants(dataset_name, settings.imbalance_levels):
            for model in settings.model_choices:
                print(f"Computing XAI metrics: {dataset_variant} | {model}")

                shap_path = (
                    shap_values_dir(settings, dataset_name, model, dataset_variant)
                    / f"{dataset_variant}_{model}_shap_values.csv"
                )

                if not shap_path.exists():
                    print(f"[SKIP] SHAP file not found: {shap_path}")
                    continue

                df_shap = pd.read_csv(shap_path)

                if not has_required_columns(
                    df_shap,
                    {"n_iteration", "index"},
                    context=str(shap_path),
                ):
                    continue

                df_detail_local, df_resume_local, df_resume_global = compute_xai_metrics_from_shap(df_shap)

                out = metrics_dir(settings, dataset_name, model, dataset_variant)

                save_csv(
                    df_detail_local,
                    out / f"{dataset_variant}_{model}_detail_local.csv",
                )
                save_csv(
                    df_resume_local,
                    out / f"{dataset_variant}_{model}_resume_local.csv",
                )
                save_csv(
                    df_resume_global,
                    out / f"{dataset_variant}_{model}_resume_global.csv",
                )


# =============================================================================
# 2. Performance summaries and rankings
# =============================================================================

def compute_resume_performance(
    df_perf: pd.DataFrame,
    dataset_name: str,
    dataset_variant: str,
    model: str,
) -> pd.DataFrame:
    imbalance = get_imbalance_label(dataset_name, dataset_variant)

    resume = {
        "dataset": dataset_name,
        "model": model,
        "imbalance": imbalance,
        "n_iterations": df_perf["n_iteration"].nunique(),
        "n_pos_sampled": df_perf["n_pos_sampled"].median(),
        "n_neg_sampled": df_perf["n_neg_sampled"].median(),
    }

    for metric in ["pr_auc", "pg", "ks"]:
        resume[f"avg_{metric}"] = df_perf[metric].mean()
        resume[f"std_{metric}"] = df_perf[metric].std()
        resume[f"median_{metric}"] = df_perf[metric].median()
        resume[f"var_{metric}"] = df_perf[metric].var()

    return pd.DataFrame([resume])


def find_performance_file(settings: Settings, dataset_name: str, model: str, dataset_variant: str) -> Path | None:
    base_dir = shap_values_dir(settings, dataset_name, model, dataset_variant)

    candidate_paths = [
        base_dir / f"{dataset_variant}_{model}_performance.csv",
        base_dir / f"{dataset_variant}_{model}_perfomance.csv",  # legacy typo
    ]

    for path in candidate_paths:
        if path.exists():
            return path

    return None


def collect_performance_data(settings: Settings) -> tuple[pd.DataFrame, pd.DataFrame]:
    summaries = []
    details = []
    required_cols = {
        "n_iteration",
        "n_pos_sampled",
        "n_neg_sampled",
        "pr_auc",
        "pg",
        "ks",
    }

    for dataset_name in settings.datasets_to_run:
        print_dataset_header("Performance", dataset_name)

        for dataset_variant in dataset_variants(dataset_name, settings.imbalance_levels):
            for model in settings.model_choices:
                print(f"Reading performance: {dataset_variant} | {model}")

                perf_path = find_performance_file(settings, dataset_name, model, dataset_variant)

                if perf_path is None:
                    print("[SKIP] Performance file not found")
                    continue

                df_perf = pd.read_csv(perf_path)

                if not has_required_columns(df_perf, required_cols, context=str(perf_path)):
                    continue

                df_perf_details = df_perf.copy()
                df_perf_details["dataset"] = dataset_name
                df_perf_details["model"] = model
                df_perf_details["imbalance"] = get_imbalance_label(dataset_name, dataset_variant)
                df_perf_details["dataset_variant"] = dataset_variant
                details.append(df_perf_details)

                df_resume = compute_resume_performance(
                    df_perf=df_perf,
                    dataset_name=dataset_name,
                    dataset_variant=dataset_variant,
                    model=model,
                )

                save_csv(
                    df_resume,
                    metrics_dir(settings, dataset_name, model, dataset_variant)
                    / f"{dataset_variant}_{model}_resume_performance.csv",
                )

                summaries.append(df_resume)

    if not summaries:
        raise ValueError("No performance summaries were created.")

    return (
        pd.concat(summaries, ignore_index=True),
        pd.concat(details, ignore_index=True),
    )


def add_performance_rankings(df_full: pd.DataFrame) -> pd.DataFrame:
    rank_config = {
        "avg_pr_auc": False,
        "avg_pg": False,
        "avg_ks": False,
        "std_pr_auc": True,
        "std_pg": True,
        "std_ks": True,
    }

    df = rank_within_groups(
        df=df_full,
        group_cols=["dataset", "imbalance"],
        rank_config=rank_config,
    )

    rank_cols = [f"rank_{metric}" for metric in rank_config]
    df["avg_rank"] = df[rank_cols].mean(axis=1)
    df["final_rank"] = (
        df.groupby(["dataset", "imbalance"])["avg_rank"]
        .rank(method="dense", ascending=True)
    )

    return df


def create_dataset_model_summary(
    df_ranked: pd.DataFrame,
    avg_rank_col: str = "avg_rank",
    final_rank_col: str = "final_rank_dataset",
) -> pd.DataFrame:
    rank_cols = [c for c in df_ranked.columns if c.startswith("rank_")]

    df_summary = (
        df_ranked
        .groupby(["dataset", "model"], as_index=False)[rank_cols + [avg_rank_col]]
        .mean()
    )

    df_summary[final_rank_col] = (
        df_summary.groupby("dataset")[avg_rank_col]
        .rank(method="dense", ascending=True)
    )

    return df_summary


def create_model_summary(
    df_dataset_model: pd.DataFrame,
    avg_rank_col: str = "avg_rank",
    final_rank_col: str = "final_model_rank",
) -> pd.DataFrame:
    rank_cols = [c for c in df_dataset_model.columns if c.startswith("rank_")]

    df_model = (
        df_dataset_model
        .groupby("model", as_index=False)[rank_cols + [avg_rank_col]]
        .mean()
    )

    df_model[final_rank_col] = df_model[avg_rank_col].rank(method="dense", ascending=True)

    return df_model


def run_performance_pipeline(settings: Settings) -> None:
    out = output_dir(settings)

    df_full, df_details = collect_performance_data(settings)
    save_csv(df_details, out / "resume_performance_details.csv")
    save_csv(df_full, out / "resume_performance_rank.csv")

    df_full_ranked = add_performance_rankings(df_full)
    save_csv(df_full_ranked, out / "resume_performance_model_dataset_imbalance_rank.csv")

    df_dataset_model = create_dataset_model_summary(df_full_ranked)
    save_csv(df_dataset_model, out / "resume_performance_model_dataset_rank.csv")

    df_model = create_model_summary(df_dataset_model)
    save_csv(df_model, out / "resume_performance_model_rank.csv")


# =============================================================================
# 3. Merge XAI local/global result files
# =============================================================================

def merge_xai_results(settings: Settings) -> dict[str, dict[str, dict[str, pd.DataFrame | None]]]:
    results = {}

    for dataset_name in settings.datasets_to_run:
        print_dataset_header("Merge XAI results", dataset_name)

        dataset_xai_dir = dataset_output_dir(settings, dataset_name) / "xai"
        results[dataset_name] = {}

        for file_type in settings.xai_file_types:
            dfs = []
            print(f"Merging file type: {file_type}")

            for imbalance in settings.imbalance_levels:
                dataset_variant = f"{dataset_name}_{imbalance}"

                for model in settings.model_choices:
                    path = (
                        dataset_xai_dir
                        / model
                        / dataset_variant
                        / "metrics"
                        / f"{dataset_variant}_{model}_{file_type}.csv"
                    )

                    if not path.exists():
                        print(f"[SKIP] Missing file: {path}")
                        continue

                    df = pd.read_csv(path)
                    df["dataset"] = dataset_name
                    df["imbalance"] = imbalance
                    df["model"] = model
                    dfs.append(df)

            if not dfs:
                print(f"[SKIP] No files found for dataset={dataset_name}, file_type={file_type}")
                results[dataset_name][file_type] = {"raw": None, "normalized": None}
                continue

            df_raw = pd.concat(dfs, ignore_index=True)
            raw_path = dataset_xai_dir / f"{dataset_name}_xai_all_models_{file_type}.csv"
            save_csv(df_raw, raw_path)

            df_normalized = None
            if settings.normalize_xai_outputs:
                df_normalized = normalize_numeric_columns(
                    df_raw,
                    exclude_cols={"imbalance", "feature", "index", "feature_importance"},
                )
                normalized_path = dataset_xai_dir / f"{dataset_name}_xai_all_models_{file_type}_normalized.csv"
                save_csv(df_normalized, normalized_path)

            results[dataset_name][file_type] = {
                "raw": df_raw,
                "normalized": df_normalized,
            }

    return results


def normalize_numeric_columns(df: pd.DataFrame, exclude_cols: set[str]) -> pd.DataFrame:
    df_normalized = df.copy()
    numeric_cols = df_normalized.select_dtypes(include="number").columns.tolist()
    numeric_cols = [col for col in numeric_cols if col not in exclude_cols]

    for col in numeric_cols:
        min_val = df_normalized[col].min()
        max_val = df_normalized[col].max()

        if pd.isna(min_val) or pd.isna(max_val):
            continue

        if max_val > min_val:
            df_normalized[col] = (df_normalized[col] - min_val) / (max_val - min_val)
        else:
            df_normalized[col] = 0.0

    return df_normalized


# =============================================================================
# 4. Stability summaries and rankings
# =============================================================================

def compute_resume_local_stability(df_local: pd.DataFrame) -> pd.DataFrame:
    metric_cols = ["rankstab_local", "rankstab_local_weight"]
    return df_local.groupby(["dataset", "model", "imbalance"], as_index=False)[metric_cols].mean()


def compute_resume_global_stability(df_global: pd.DataFrame) -> pd.DataFrame:
    metric_cols = [
        "rankstab_global",
        "value_stab",
        "value_stab_weight",
        "rankstab_global_weight",
    ]
    return df_global.groupby(["dataset", "model", "imbalance"], as_index=False)[metric_cols].mean()


def collect_stability_summaries(settings: Settings) -> tuple[pd.DataFrame, pd.DataFrame]:
    local_summaries = []
    global_summaries = []

    required_local_cols = {
        "dataset",
        "imbalance",
        "model",
        "rankstab_local",
        "rankstab_local_weight",
    }
    required_global_cols = {
        "dataset",
        "imbalance",
        "model",
        "rankstab_global",
        "value_stab",
        "value_stab_weight",
        "rankstab_global_weight",
    }

    for dataset_name in settings.datasets_to_run:
        print_dataset_header("Stability", dataset_name)

        dataset_xai_dir = dataset_output_dir(settings, dataset_name) / "xai"
        local_path = dataset_xai_dir / f"{dataset_name}_xai_all_models_resume_local.csv"
        global_path = dataset_xai_dir / f"{dataset_name}_xai_all_models_resume_global.csv"

        if local_path.exists():
            df_local = pd.read_csv(local_path)
            if has_required_columns(df_local, required_local_cols, context=str(local_path)):
                local_summaries.append(compute_resume_local_stability(df_local))
        else:
            print(f"[SKIP] Local stability file not found: {local_path}")

        if global_path.exists():
            df_global = pd.read_csv(global_path)
            if has_required_columns(df_global, required_global_cols, context=str(global_path)):
                global_summaries.append(compute_resume_global_stability(df_global))
        else:
            print(f"[SKIP] Global stability file not found: {global_path}")

    if not local_summaries:
        raise ValueError("No local stability files were loaded.")
    if not global_summaries:
        raise ValueError("No global stability files were loaded.")

    return (
        pd.concat(local_summaries, ignore_index=True),
        pd.concat(global_summaries, ignore_index=True),
    )


def add_stability_rankings(df_stability: pd.DataFrame) -> pd.DataFrame:
    local_rank_config = {
        "rankstab_local": False,
        "rankstab_local_weight": False,
    }
    global_rank_config = {
        "rankstab_global": True,
        "value_stab": True,
        "value_stab_weight": True,
        "rankstab_global_weight": True,
    }

    df = rank_within_groups(
        df=df_stability,
        group_cols=["dataset", "imbalance"],
        rank_config=local_rank_config | global_rank_config,
    )

    rank_cols = [
        f"rank_{metric}"
        for metric in list(local_rank_config) + list(global_rank_config)
    ]

    df["avg_rank_stability"] = df[rank_cols].mean(axis=1)
    df["final_rank_stability"] = (
        df.groupby(["dataset", "imbalance"])["avg_rank_stability"]
        .rank(method="dense", ascending=True)
    )

    return df


def run_stability_pipeline(settings: Settings) -> None:
    out = output_dir(settings)

    df_local, df_global = collect_stability_summaries(settings)

    df_stability = pd.merge(
        df_local,
        df_global,
        on=["dataset", "model", "imbalance"],
        how="outer",
    )

    df_stability = add_stability_rankings(df_stability)
    save_csv(df_stability, out / "resume_stability_model_dataset_imbalance_rank.csv")

    df_dataset_model = create_dataset_model_summary(
        df_ranked=df_stability,
        avg_rank_col="avg_rank_stability",
        final_rank_col="final_rank_dataset_stability",
    )
    save_csv(df_dataset_model, out / "resume_stability_model_dataset_rank.csv")

    df_model = create_model_summary(
        df_dataset_model=df_dataset_model,
        avg_rank_col="avg_rank_stability",
        final_rank_col="final_model_rank_stability",
    )
    save_csv(df_model, out / "resume_stability_model_rank.csv")


# =============================================================================
# 5. Merge performance and stability rankings
# =============================================================================

def merge_rankings(
    perf_path: Path,
    stab_path: Path,
    merge_cols: list[str],
    group_cols_for_final_rank: list[str] | None,
    save_path: Path,
) -> pd.DataFrame:
    df_perf = pd.read_csv(perf_path)
    df_stab = pd.read_csv(stab_path)

    df = pd.merge(
        df_perf,
        df_stab,
        on=merge_cols,
        how="inner",
        suffixes=("_performance", "_stability"),
    )

    df["avg_rank_combined"] = (df["avg_rank"] + df["avg_rank_stability"]) / 2

    if group_cols_for_final_rank:
        df["final_rank_combined"] = (
            df.groupby(group_cols_for_final_rank)["avg_rank_combined"]
            .rank(method="dense", ascending=True)
        )
    else:
        df["final_rank_combined"] = df["avg_rank_combined"].rank(method="dense", ascending=True)

    save_csv(df, save_path)
    return df


def run_combined_ranking_pipeline(settings: Settings) -> None:
    out = output_dir(settings)

    merge_rankings(
        perf_path=out / "resume_performance_model_rank.csv",
        stab_path=out / "resume_stability_model_rank.csv",
        merge_cols=["model"],
        group_cols_for_final_rank=None,
        save_path=out / "resume_model_rank_merged.csv",
    )

    merge_rankings(
        perf_path=out / "resume_performance_model_dataset_rank.csv",
        stab_path=out / "resume_stability_model_dataset_rank.csv",
        merge_cols=["dataset", "model"],
        group_cols_for_final_rank=["dataset"],
        save_path=out / "resume_model_dataset_rank_merged.csv",
    )

    merge_rankings(
        perf_path=out / "resume_performance_model_dataset_imbalance_rank.csv",
        stab_path=out / "resume_stability_model_dataset_imbalance_rank.csv",
        merge_cols=["dataset", "model", "imbalance"],
        group_cols_for_final_rank=["dataset", "imbalance"],
        save_path=out / "resume_model_dataset_imbalance_rank_merged.csv",
    )


# =============================================================================
# 6. Orchestration
# =============================================================================

def print_dataset_header(stage: str, dataset_name: str) -> None:
    print("\n" + "#" * 90)
    print(f"{stage.upper()} | DATASET: {dataset_name}")
    print("#" * 90)


def run_pipeline(settings: Settings) -> None:
    """
    Main entry point.

    The only object you should normally edit is Settings.
    """
    compute_and_save_xai_metrics(settings)
    run_performance_pipeline(settings)
    merge_xai_results(settings)
    run_stability_pipeline(settings)
    run_combined_ranking_pipeline(settings)
    print("\nDONE ✅ All XAI, performance, stability, and combined ranking files saved.")


if __name__ == "__main__":
    settings = Settings()
    run_pipeline(settings)



##########################################################################################
XAI METRICS | DATASET: australian
##########################################################################################
Computing XAI metrics: australian | XGBoost
[SKIP] SHAP file not found: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/xai/XGBoost/australian/shap_values/australian_XGBoost_shap_values.csv
Computing XAI metrics: australian | LightGBM
[SKIP] SHAP file not found: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/xai/LightGBM/australian/shap_values/australian_LightGBM_shap_values.csv
Computing XAI metrics: australian | RandomForest
[SKIP] SHAP file not found: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/xai/RandomForest/australian/shap_values/australian_RandomForest_shap_values.csv
Computing XAI metrics: australian | BoostDT
[SKIP] SHAP file not found: /Users/eduardohome/workspace/benchma